---
title: "L6-04 · Operational interventions"
short_title: "Operational interventions"
subtitle: "In a quantum process there is no passive look: every operation at a node is an instrument, and an input-independent replacement is the operation that cuts the incoming arrow the way Pearl's do does."
description: "Distinguish conditioning on a measurement outcome, measuring and discarding the outcome, and replacing a system with a fresh state, and compute each one on a two-node quantum process."
tags: [level-6, coursebook]
---

(lesson-l6-04)=

:::{div}
:class: qcai-meta
![Level 6 icon](../../_static/icons/qicon-level-6.svg) **Level 6 · Graduate** · about 110 minutes  
**Big idea.** In a quantum process there is no passive look: every operation at a node is an instrument, and an input-independent replacement is the operation that cuts the incoming arrow the way Pearl's do does.  
**Before this lesson.** [L6-03](#lesson-l6-03), [L5-07](#lesson-l5-07), [L4-01](#lesson-l4-01)  
**Lab.** [Confounding and intervention](../../labs/03-causal-inference.ipynb)
:::

## What you will be able to do

- Write the three operations (Lüders conditioning, non-selective measurement, replacement) as instruments and compute what each does to a downstream system.
- Prove that a channel's output is independent of its input exactly when its Choi operator is $I \otimes \sigma$.
- On a mixture of a common cause and a direct cause with weight $p$, compute $p(b=0 \mid a=0) = 1$, the interventional values $1 - p/2$ and $p/2$, and the Bloch-sphere images of each operation.
- Explain why selecting outcomes does not identify a causal effect, and why measuring and discarding is not the same as doing nothing.

## Why this matters

Classical causal inference rests on one contrast: seeing $X = x$ is not the same as setting $X$ to $x$. In [L4-01](#lesson-l4-01) and the do-calculus of [L4-04](#lesson-l4-04), seeing is free; it never disturbs the system. Quantum systems remove that luxury. To see the value of a qubit you must measure it, and the measurement changes what flows downstream. So the quantum version of the contrast has at least three terms, not two.

Here is the failure that motivates the lesson. In the process below, a qubit at A and a qubit at B agree perfectly whenever A measures $Z$: $p(b=0 \mid a=0) = 1$. A naive analyst concludes that A fully controls B. Yet if A throws its qubit away and prepares $|0\rangle$, B shows 0 only three times in four. Both numbers are correct; they answer different questions. This lesson builds the vocabulary that keeps them apart.

:::{dropdown} Key terms

**Instrument**
: A collection $\{\mathcal M_a\}$ of completely positive maps whose sum $\sum_a \mathcal M_a$ is trace preserving. Outcome $a$ occurs with probability $\operatorname{Tr}\mathcal M_a(\rho)$ and leaves $\mathcal M_a(\rho)/\operatorname{Tr}\mathcal M_a(\rho)$.

**Lüders measurement**
: The instrument $\mathcal M_a(\rho) = \Pi_a \rho\, \Pi_a$ for a projective measurement $\{\Pi_a\}$: the system continues in the collapsed state.

**Conditioning**
: Keeping only the runs in which outcome $a$ occurred and renormalizing. It selects data; it does not change the process.

**Non-selective measurement**
: Performing an instrument and forgetting the outcome: the channel $\sum_a \mathcal M_a$.

**Replacement channel**
: $R_\sigma(\rho) = \operatorname{Tr}(\rho)\,\sigma$: discard the input and prepare $\sigma$. Its output does not depend on the input.

**Choi operator**
: $J_{\mathcal E} = \sum_{ij} |i\rangle\langle j| \otimes \mathcal E(|i\rangle\langle j|)$, with $\mathcal E(\rho) = \operatorname{Tr}_{in}[(\rho^{T} \otimes I)J_{\mathcal E}]$ (the convention of [L6-02](#lesson-l6-02)).

:::


In [1]:
import sys, pathlib, subprocess
here = pathlib.Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "qcai").is_dir()), None)
if root is None:  # for example on Google Colab: fetch the course's small teaching package
    root = pathlib.Path("quantum-causal-ai")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pozapas/quantum-causal-ai.git", str(root)], check=True)
    root = root.resolve()
sys.path.insert(0, str(root))
from qcai.viz import show as _show, show_mpl as _show_mpl
from qcai.lessonfigs import l6_04 as _lessonfigs

## Three operations at a node

In process-based quantum causal models, a node is a laboratory with an input system and an output system, and what an agent does there is the choice of an instrument [@costa]. Fix a projective measurement $\{\Pi_a\}$ on A's input. Three different things can happen to the data.

:::{prf:definition} Conditioning, measuring and discarding, replacing
:label: def-l6-04-ops
Let $\rho$ be the joint state of A's input and everything else.

1. **Conditioning on $a$** (Lüders): apply $\mathcal M_a(\cdot) = \Pi_a(\cdot)\Pi_a$, keep only runs with outcome $a$, and renormalize by $p(a) = \operatorname{Tr}[(\Pi_a \otimes I)\rho]$.
2. **Measuring and discarding**: apply the channel $\mathcal D = \sum_a \mathcal M_a$ and keep every run.
3. **Replacing by $\sigma$**: apply $R_\sigma(\tau) = \operatorname{Tr}(\tau)\,\sigma$ to A's input, so A's output is $\sigma$ whatever came in. We write $p(\cdot \mid do(\sigma))$ for statistics under this operation.
:::

$$
\sum_a p(a)\, \rho_{B \mid a} = \rho_B^{\mathcal D}
$$ (eq-l6-04-total)

Equation [](#eq-l6-04-total) is the quantum law of total probability for instruments: averaging the conditional states downstream with the outcome probabilities gives the state under the non-selective operation. It follows from linearity, because $\sum_a \mathcal M_a = \mathcal D$. What it does **not** say is that $\rho_B^{\mathcal D}$ equals the state with no operation at all; in general it does not.

:::{prf:example} No passive look
:label: ex-l6-04-nolook
A prepares $|+\rangle$ and sends it straight to B, who measures $X$. If A does nothing, B always sees $+$. If A measures $Z$ and discards the outcome, A's output is $\tfrac12(|0\rangle\langle 0| + |1\rangle\langle 1|) = I/2$, and B sees $+$ only half the time. The act of looking, even without reading the result, changed B's statistics. In a classical model, observing a variable never changes its children.
:::

## One process, three answers

We use one process throughout, the probabilistic mixture of a common cause and a direct cause studied by Ried and coauthors [@ried2015]. A source prepares $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt2$ on A's input $A_{in}$ and a side system $E$. With probability $p$, B receives $E$ and A's output is lost (common cause). With probability $1 - p$, B receives A's output $A_{out}$ unchanged and $E$ is lost (direct cause). B measures $Z$. A's input is maximally mixed in both branches.

```{mermaid}
graph LR
  S[source Phi+] -->|A_in| A[lab A]
  S -->|E| M{mix p}
  A -->|A_out| M
  M --> B[lab B]
  F[fresh state] -.->|do| A
```

*The two-node process. The mixing box sends $E$ to B with probability $p$ (common cause) and A's output with probability $1 - p$ (direct cause). The dotted arrow is the replacement intervention: A discards $A_{in}$ and sends a fresh state instead.*

:::{prf:proposition} The four numbers
:label: prop-l6-04-numbers
In this process, with B measuring $Z$ and A using the $Z$ basis:

$$p(b=0 \mid a=0) = 1, \qquad p(b=0)\big|_{\mathcal D} = \tfrac12, \qquad p(b=0 \mid do(|0\rangle)) = 1 - p/2, \qquad p(b=0 \mid do(|1\rangle)) = p/2.$$

The observed association $p(b=0 \mid a=0) - p(b=0 \mid a=1)$ is 1 for every $p$, while the average causal effect $p(b=0 \mid do(|0\rangle)) - p(b=0 \mid do(|1\rangle))$ is $1 - p$. At $p = 1/2$ the interventional values are $3/4$ and $1/4$.
:::

:::{prf:proof}
Conditioning: the Lüders measurement with outcome 0 (probability $1/2$) leaves $|0\rangle|0\rangle$ on $A_{in}E$, since $(\Pi_0 \otimes I)|\Phi^+\rangle = |00\rangle/\sqrt2$. A's output is $|0\rangle$ and so is $E$, so both branches deliver $|0\rangle$ to B. Measuring and discarding: the state becomes $\tfrac12(|00\rangle\langle 00| + |11\rangle\langle 11|)$, whose marginals are $I/2$, so B gets $I/2$ in both branches. Replacement by $|0\rangle$: A's output is $|0\rangle$, while $E$ keeps its marginal $I/2$; B gets $p\, I/2 + (1 - p)|0\rangle\langle 0|$, so $p(b=0) = p/2 + 1 - p = 1 - p/2$. The same computation with $|1\rangle$ gives $p/2$.
:::

:::{figure} ../../_static/animations/l6_04_three_operations.mp4
:alt: The process diagram with source, lab A, the mixing box and lab B; then three panels: conditioning gives 1, measuring and discarding gives one half, and replacing A's input by a fresh zero state gives three quarters, with one quarter for a fresh one state.
:width: 100%

Three operations at A on one process with $p = 1/2$: conditioning gives $p(b=0 \mid a=0) = 1$, measuring and discarding gives $1/2$, and replacement gives $3/4$ and $1/4$.
:::

:::{dropdown} What the animation shows
The animation draws the process: a source prepares $|\Phi^+\rangle$ and sends $A_{in}$ to lab A and $E$ to a mixing box; A's output $A_{out}$ also enters the mixing box; the box passes one of them to lab B. A note says that with probability $p$ B gets $E$ (common cause) and with probability $1 - p$ B gets A's output (direct cause), with $p = 1/2$ and B measuring $Z$. Panel 1, condition: A measures $Z$ and keeps $a = 0$; both branches deliver $|0\rangle$, so $p(b=0 \mid a=0) = 1$. Panel 2, measure and discard: $p(b=0) = 1/2$. Panel 3, replace: a red cross cuts the wire $A_{in}$ and a fresh $|0\rangle$ enters A; then $p(b=0 \mid do(|0\rangle)) = 1 - p/2 = 3/4$ and $p(b=0 \mid do(|1\rangle)) = p/2 = 1/4$. The verdict: the association is $1 - 0 = 1$, but the causal effect is $3/4 - 1/4 = 1 - p$.
:::

:::{warning} Selection is not intervention
Conditioning picks the runs in which A's outcome was 0, and in those runs the common cause also delivered 0 to B. That is exactly the confounding that do removes. The gap $p(b=0 \mid a=0) - p(b=0 \mid do(|0\rangle)) = p/2$ is the quantum analogue of confounding bias, and it disappears only when $p = 0$.
:::

## Replacement cuts the arrow

:::{prf:theorem} Input independence is a product Choi operator
:label: thm-l6-04-product
A channel $\mathcal E$ from $\mathbb C^d$ to $\mathbb C^{d'}$ satisfies $\mathcal E(\rho) = \sigma$ for every state $\rho$ if and only if $J_{\mathcal E} = I \otimes \sigma$. In that case $\mathcal E = R_\sigma$.
:::

:::{prf:proof}
If $J_{\mathcal E} = I \otimes \sigma$, the recovery rule gives $\mathcal E(\rho) = \operatorname{Tr}_{in}[(\rho^{T} \otimes I)(I \otimes \sigma)] = \operatorname{Tr}(\rho^{T})\,\sigma = \operatorname{Tr}(\rho)\,\sigma$. Conversely, suppose $\mathcal E(\rho) = \sigma$ for all states. Density operators span all $d \times d$ matrices, and every matrix $X$ is a combination $\sum_k c_k \rho_k$ of states with $\sum_k c_k = \operatorname{Tr} X$. By linearity $\mathcal E(X) = \operatorname{Tr}(X)\,\sigma$, so $\mathcal E(|i\rangle\langle j|) = \delta_{ij}\sigma$ and $J_{\mathcal E} = \sum_{ij}|i\rangle\langle j| \otimes \delta_{ij}\sigma = I \otimes \sigma$ [@watrous].
:::

:::{figure} ../../_static/animations/l6_04_replacement_cut.mp4
:alt: A derivation showing that the replacement channel with Choi operator I tensor sigma outputs Tr(rho) sigma, then a graph in which the conditional probability stays at 1 while the interventional probabilities move along 1 minus p over 2 and p over 2 as the common-cause weight p goes from 0 to 1.
:width: 100%

The product Choi operator $J_R = I\otimes\sigma$ cuts the input; then a sweep of the common-cause weight $p$ separates conditioning from intervention.
:::

:::{dropdown} What the animation shows
The convention $\mathcal E(\rho) = \operatorname{Tr}_{in}[(\rho^{T} \otimes I)J_{\mathcal E}]$ is written, then $J_R = I\otimes\sigma$. The derivation transforms $R_\sigma(\rho) = \operatorname{Tr}_{in}[(\rho^{T} \otimes I)(I \otimes \sigma)]$ into $\operatorname{Tr}(\rho^{T})\sigma$ and then $\operatorname{Tr}(\rho)\sigma$: the output never depends on the input. Next, axes show $p(b=0)$ against the common-cause weight $p$ from 0 to 1. Three points move as $p$ increases and leave traces: blue for $p(b=0 \mid a=0) = 1$, flat; red for $p(b=0 \mid do(|0\rangle)) = 1 - p/2$, falling from 1 to $1/2$; amber for $p(b=0 \mid do(|1\rangle)) = p/2$, rising from 0 to $1/2$. A dashed line at $p = 1/2$ marks 1 against $3/4$: conditioning exceeds the do value by $p/2$.
:::

:::{prf:proposition} Unselected local operations do not signal through a common cause
:label: prop-l6-04-nosignal
If A and B share a state $\rho_{AB}$ and nothing else connects them, then for every channel $\mathcal E$ applied at A, B's state is unchanged: $\operatorname{Tr}_A[(\mathcal E \otimes \mathrm{id})(\rho_{AB})] = \operatorname{Tr}_A \rho_{AB}$.
:::

:::{prf:proof}
Write $\rho_{AB} = \sum_k X_k \otimes Y_k$. Then $\operatorname{Tr}_A[(\mathcal E \otimes \mathrm{id})\rho_{AB}] = \sum_k \operatorname{Tr}[\mathcal E(X_k)]\,Y_k = \sum_k \operatorname{Tr}(X_k)\,Y_k = \operatorname{Tr}_A\rho_{AB}$, because $\mathcal E$ preserves trace. So in the pure common-cause case ($p = 1$), every replacement gives $p(b=0 \mid do(\sigma)) = 1/2$: the causal effect $1 - p$ is 0, as it should be.
:::

:::{prf:remark} Other interventions
:label: rem-l6-04-other
Replacement is the operation that matches Pearl's do: it removes every dependence of A's output on what came in [@pearl]. It is not the only useful intervention. Ried and coauthors measure the early system and then reprepare a state drawn at random from a tomographically complete set; this randomized repreparation plays the role of a randomized trial and allows full reconstruction of the process, which they call causal tomography [@ried2015]. In general the set of instruments available at each node is part of the model, and conclusions can change when an operation is unavailable [@costa].
:::

## The three operations on the Bloch sphere

:::{prf:proposition} Bloch images at B
:label: prop-l6-04-bloch
In the process above, write Bloch vectors as $\vec r$. (i) Replacement by $\sigma$ gives $\vec r_B = (1-p)\,\vec r_\sigma$. (ii) Conditioning on outcome $+\vec n$ of a Lüders measurement of $\vec n \cdot \vec\sigma$ gives $\vec r_B = (n_x, (1-2p)n_y, n_z)$. (iii) Measuring and discarding, in any basis, gives $\vec r_B = \vec 0$.
:::

:::{prf:proof}
(i) The common-cause branch delivers $E$, whose marginal is $I/2$; the direct branch delivers $\sigma$. So $\rho_B = p\, I/2 + (1-p)\sigma$. (ii) For $|\Phi^+\rangle$, $(\Pi \otimes I)|\Phi^+\rangle = (I \otimes \Pi^{T})|\Phi^+\rangle$. With $\Pi = \tfrac12(I + \vec n\cdot\vec\sigma)$ and $X^{T} = X$, $Y^{T} = -Y$, $Z^{T} = Z$, the outcome $+\vec n$ has probability $1/2$ and leaves $E$ with Bloch vector $(n_x, -n_y, n_z)$. The direct branch carries the collapsed state with Bloch vector $\vec n$. The mixture gives $p(n_x, -n_y, n_z) + (1-p)\vec n = (n_x, (1-2p)n_y, n_z)$. (iii) Averaging (ii) over $\pm\vec n$ with weights $1/2$ gives $\vec 0$.
:::

:::{figure} ../../_static/animations/l6_04_bloch_images.mp4
:alt: A rotating Bloch sphere covered by forty-eight colored dots; replacement shrinks the dots to a ball of half the radius; conditioning flattens them into a disk at p one half and mirrors them at p equal to one; measuring and discarding collapses all of them to the center.
:width: 100%

Replacement shrinks the sphere to radius $1 - p$; conditioning maps it to $(n_x, (1-2p)n_y, n_z)$, flat at $p = 1/2$ and mirrored at $p = 1$; measuring and discarding sends everything to the center.
:::

:::{dropdown} What the animation shows
Forty-eight dots cover the Bloch sphere, blue where $y > 0$ and amber where $y < 0$; they stand for A's options. First, replacement: the formula $do(\sigma): \vec r_B = (1-p)\,\vec r_\sigma$ appears and the dots shrink to a sphere of radius $1/2$ ($p = 1/2$): interventions see only the direct-cause branch. The dots return. Second, conditioning on $+\vec n$ with $\vec r_B = (n_x, (1-2p)n_y, n_z)$: as $p$ runs from 0 to $1/2$ the sphere flattens into a disk in the $x$-$z$ plane, because the $y$-components of the two branches cancel; as $p$ continues to 1 it reopens with blue and amber swapped, a mirror image in $y$ (pure common cause). Third, measuring and discarding: every dot moves to the center, $\vec r_B = \vec 0$.
:::

Part (ii) is the seed of the next lesson. With A's input maximally mixed, B's conditional states reveal the sign of the $y$-component: the direct branch preserves it and the common-cause branch flips it, because $Y^{T} = -Y$. Ried and coauthors connect this kind of signature to quantum conditional states in the framework of Leifer and Spekkens [@ried2015; @leifer]; in this course it is the transpose in the recovery formula of [L6-02](#lesson-l6-02) at work. In [L6-05](#lesson-l6-05) it becomes a tool for causal discovery without intervention.


In [2]:
_show(_lessonfigs.operations_explorer())

**Explore.** Move the common-cause weight $p$. Left: B's probability of 0 under each operation at A. Right: the section $z = 0$ of B's Bloch images for conditioning (blue ellipse) and replacement (red circle).

:::{dropdown} What the figure shows
The first panel shows four bars: conditioning on $a = 0$ (always 1), measuring and discarding (always $1/2$), $do(|0\rangle)$ at $1 - p/2$ and $do(|1\rangle)$ at $p/2$. The second panel shows the unit circle of A's options in the $x$-$y$ plane, its image under conditioning, an ellipse with semi-axes 1 and $|1-2p|$, its image under replacement, a circle of radius $1 - p$, and the single point at the origin for measuring and discarding. At $p = 0$ the blue and red curves coincide with the unit circle; at $p = 1/2$ the ellipse is a flat segment; at $p = 1$ the red circle has shrunk to the origin.
:::

## Compute it

Simulate the process with density matrices: the Lüders, non-selective and replacement instruments at A, the mixing channel, B's statistics, the law of total probability [](#eq-l6-04-total), and the product Choi operator of a replacement channel (using `qcai.science.choi`).


In [3]:
import math
import numpy as np
from qcai import science as s

# The two-node process.  Order of systems: A_in (A's input), E (side system), A_out (A's output).
# Source |Phi+> on (A_in, E).  With prob. p, B receives E (common cause); with prob. 1 - p, B receives A_out.
phi = s.bell_state("phi+")
rho_src = s.dm(phi)                                  # on A_in (x) E
P0, P1 = s.dm(s.KET0), s.dm(s.KET1)

def to_B(rho_AoutE, p):
    """Mixing channel: keep E with probability p, keep A_out with probability 1 - p."""
    return p * s.partial_trace(rho_AoutE, keep=1) + (1 - p) * s.partial_trace(rho_AoutE, keep=0)

def run(instrument, p):
    """instrument: list of Kraus lists, one per outcome, acting A_in -> A_out. Returns [(prob, rho_B)]."""
    out = []
    for kraus in instrument:
        r = sum(np.kron(K, s.I2) @ rho_src @ np.kron(K, s.I2).conj().T for K in kraus)
        pa = np.trace(r).real
        out.append((pa, to_B(r / pa, p) if pa > 1e-12 else None))
    return out

p = 0.5
luders_Z = [[P0], [P1]]                                       # measure Z, pass the collapsed state on
nonselective_Z = [[P0, P1]]                                   # same Kraus operators, outcome forgotten
replace = lambda ket: [[np.outer(ket, e) for e in (s.KET0, s.KET1)]]   # discard input, prepare |ket>

(pa0, B_a0), (pa1, B_a1) = run(luders_Z, p)
[(_, B_ns)] = run(nonselective_Z, p)
[(_, B_do0)] = run(replace(s.KET0), p)
[(_, B_do1)] = run(replace(s.KET1), p)
print(f"p(a=0) = {pa0:.2f}   p(b=0 | a=0) = {B_a0[0,0].real:.2f}   p(b=0 | a=1) = {B_a1[0,0].real:.2f}")
print(f"measure and discard: p(b=0) = {B_ns[0,0].real:.2f}")
print(f"do(|0>): p(b=0) = {B_do0[0,0].real:.2f}   do(|1>): p(b=0) = {B_do1[0,0].real:.2f}")
print("average causal effect 1 - p =", round(B_do0[0,0].real - B_do1[0,0].real, 4))

# Non-selective = average of the conditional states (law of total probability for instruments)
assert np.allclose(B_ns, pa0 * B_a0 + pa1 * B_a1)

# Replacement channel: Choi operator is I (x) sigma (convention J = sum |i><j| (x) E(|i><j|))
sigma = s.dm(np.array([1, 1]) / math.sqrt(2))
w, V = np.linalg.eigh(sigma)
K_rep = [math.sqrt(max(wk, 0)) * np.outer(V[:, k], e) for k, wk in enumerate(w) for e in (s.KET0, s.KET1)]
print("replacement is CPTP:", s.is_cptp(K_rep), "  J = I (x) sigma:", np.allclose(s.choi(K_rep), np.kron(s.I2, sigma)))

p(a=0) = 0.50   p(b=0 | a=0) = 1.00   p(b=0 | a=1) = 0.00
measure and discard: p(b=0) = 0.50
do(|0>): p(b=0) = 0.75   do(|1>): p(b=0) = 0.25
average causal effect 1 - p = 0.5
replacement is CPTP: True   J = I (x) sigma: True


Check the Bloch images of the proposition for $p = 1/4$, and the no-passive-look example.


In [4]:
# Bloch images at B.  Conditioning on outcome +n of a Lüders measurement of n.sigma at A:
def conditional_bloch(n, p):
    Pn = (s.I2 + n[0] * s.X + n[1] * s.Y + n[2] * s.Z) / 2
    Pm = s.I2 - Pn
    [(pa, rB), _] = run([[Pn], [Pm]], p)
    return s.bloch(rB)

for n in [(1, 0, 0), (0, 1, 0), (0, 0, 1), (0.6, 0.8, 0)]:
    r = conditional_bloch(np.array(n, float), 0.25)
    print("n =", n, " r_B =", np.round(r, 4), " formula:", np.round([n[0], (1 - 2 * 0.25) * n[1], n[2]], 4))

# Replacement by sigma: r_B = (1 - p) r_sigma
for ket in (s.ket(math.pi / 2, 0), s.ket(math.pi / 3, 1.0)):
    [(_, rB)] = run(replace(ket), 0.25)
    print("r_sigma =", np.round(s.bloch(s.dm(ket)), 4), " r_B =", np.round(s.bloch(rB), 4))

# No passive look: a product input |+>, direct cause only (p = 0), B measures X.
plus = np.array([1, 1]) / math.sqrt(2)
rho_plus = s.dm(plus)
do_nothing = s.apply_kraus(rho_plus, [s.I2])
measured = s.apply_kraus(rho_plus, [P0, P1])
pX = lambda r: np.real(np.trace(r @ s.dm(plus)))
print("p(B = +) with A doing nothing:", round(pX(do_nothing), 3), "  after A measures Z and discards:", round(pX(measured), 3))

n = (1, 0, 0)  r_B = [1. 0. 0.]  formula: [1. 0. 0.]
n = (0, 1, 0)  r_B = [0.  0.5 0. ]  formula: [0.  0.5 0. ]
n = (0, 0, 1)  r_B = [0. 0. 1.]  formula: [0. 0. 1.]
n = (0.6, 0.8, 0)  r_B = [0.6 0.4 0. ]  formula: [0.6 0.4 0. ]
r_sigma = [1. 0. 0.]  r_B = [0.75 0.   0.  ]
r_sigma = [0.4679 0.7287 0.5   ]  r_B = [0.3509 0.5466 0.375 ]
p(B = +) with A doing nothing: 1.0   after A measures Z and discards: 0.5


:::{prf:remark} Conventions
:label: rem-l6-04-conv
Choi operators use $J = \sum_{ij}|i\rangle\langle j| \otimes \mathcal E(|i\rangle\langle j|)$ with input first and the transpose in the recovery rule, as in [L6-02](#lesson-l6-02) and `qcai.science.choi`. Bloch vectors are $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$. The weight $p$ is the probability of the common-cause branch, as in [@ried2015].
:::

## Worked example

:::{admonition} The sign of the bias depends on the basis
:class: tip

Take the same process with $p = 0.3$. A uses a Lüders measurement and B measures in the same basis. Compute the conditional probability that B agrees with A's outcome $+$, and the interventional probability that B shows $+$ after A replaces its output by the $+$ eigenstate, first in the $X$ basis and then in the $Y$ basis.

1. $X$ basis: conditioning on $+\hat x$ gives $\vec r_B = (1, 0, 0)$ by the proposition, so $p(B = + \mid a = +) = (1 + 1)/2 = 1$.
2. Replacement by $|+\rangle$ gives $\vec r_B = (1 - 0.3)(1, 0, 0) = (0.7, 0, 0)$, so $p(B = + \mid do) = (1 + 0.7)/2 = 0.85$. Conditioning overstates it by $0.15 = p/2$.
3. $Y$ basis: conditioning on $+\hat y$ gives $\vec r_B = (0, 1 - 0.6, 0) = (0, 0.4, 0)$, so $p(B = + \mid a = +) = (1 + 0.4)/2 = 0.7$.
4. Replacement by the $+\hat y$ eigenstate gives $\vec r_B = (0, 0.7, 0)$, so again $0.85$. Now conditioning understates the interventional value by $0.15$.

**Result.** The interventional value $1 - p/2 = 0.85$ is the same in every basis, but the conditional value is 1 in the $X$ and $Z$ bases and $1 - p = 0.7$ in the $Y$ basis. A quantum common cause can bias conditioning upward in one basis and downward in another.
:::

## Exercises

**1. (Core)** In the process with $p = 1/2$, compute $p(b=0 \mid a=1)$ under conditioning and $p(b=0 \mid do(|1\rangle))$ under replacement.

:::{dropdown} Solution
Conditioning on $a = 1$ leaves $|1\rangle|1\rangle$ on $A_{in}E$, so both branches deliver $|1\rangle$ and $p(b=0 \mid a=1) = 0$. Replacement by $|1\rangle$ gives $\rho_B = \tfrac12 \cdot I/2 + \tfrac12|1\rangle\langle 1|$, so $p(b=0 \mid do(|1\rangle)) = p/2 = 1/4$.
:::

**2. (Core)** Write the Choi operator of the replacement channel $R_\sigma$ with $\sigma = |+\rangle\langle +|$, check $\operatorname{Tr}_{out}J = I$, and compute $R_\sigma(|1\rangle\langle 1|)$ from the recovery rule.

:::{dropdown} Solution
$J = I \otimes |+\rangle\langle +|$. Tracing out the output gives $I \cdot \operatorname{Tr}|+\rangle\langle +| = I$, so the channel is trace preserving. The recovery rule gives $\operatorname{Tr}_{in}[(|1\rangle\langle 1|^{T} \otimes I)(I \otimes |+\rangle\langle +|)] = \operatorname{Tr}(|1\rangle\langle 1|)\,|+\rangle\langle +| = |+\rangle\langle +|$.
:::

**3. (Stretch)** Prove [](#eq-l6-04-total) for an arbitrary instrument $\{\mathcal M_a\}$ at A and an arbitrary channel $\mathcal C$ from A's output and the rest of the process to B.

:::{dropdown} Solution
Let $\rho$ be the joint input. The unnormalized state after outcome $a$ is $(\mathcal M_a \otimes \mathrm{id})\rho$, with probability $p(a) = \operatorname{Tr}[(\mathcal M_a \otimes \mathrm{id})\rho]$, and $\rho_{B|a} = \mathcal C[(\mathcal M_a \otimes \mathrm{id})\rho]/p(a)$. Then $\sum_a p(a)\rho_{B|a} = \mathcal C[(\sum_a \mathcal M_a \otimes \mathrm{id})\rho] = \mathcal C[(\mathcal D \otimes \mathrm{id})\rho] = \rho_B^{\mathcal D}$ by linearity of $\mathcal C$. (Outcomes with $p(a) = 0$ contribute 0 on both sides.)
:::

**4. (Stretch)** In the process, A conditions on outcome $+\hat y$ and B measures $Y$. For which $p$ is B's outcome independent of A's? What is the average causal effect of replacing A's output by $|{+}\hat y\rangle$ versus $|{-}\hat y\rangle$ at that $p$?

:::{dropdown} Solution
Conditioning gives $\vec r_B = (0, 1 - 2p, 0)$, and conditioning on $-\hat y$ gives $(0, -(1-2p), 0)$. B's $Y$ outcome is independent of A's exactly when $1 - 2p = 0$, that is $p = 1/2$. The interventional values are $(1 \pm (1-p))/2$, so the average causal effect on $p(B = +)$ is $1 - p = 1/2$. Zero correlation coexists with a nonzero causal effect: an exact cancellation of the kind that faithfulness assumptions rule out.
:::

**5. (Challenge)** Suppose you are promised that the process is the $\Phi^+$ and identity mixture above, with unknown $p$, but you may only condition (Lüders measurements in the three Pauli bases at A, the same basis at B). Show that $p$ is identifiable and give a formula.

:::{dropdown} Solution
Define the correlation in basis $s$ as $c_s = p(\text{agree}) - p(\text{disagree})$. With A's input maximally mixed and the proposition, conditioning on $\pm\vec n$ gives B's Bloch component along $\vec n$ equal to $\pm 1$ for $\vec n = \hat x, \hat z$ and $\pm(1 - 2p)$ for $\vec n = \hat y$. Hence $c_x = c_z = 1$ and $c_y = 1 - 2p$, so $p = (1 - c_y)/2$. The $X$ and $Z$ data carry no information about $p$; only the $Y$ data do, because the common cause flips the sign of the $Y$ correlation. This is a special case of the passive-observation result of Ried et al. [@ried2015].
:::

**6. (Challenge)** A measure-and-prepare channel is $\mathcal E(\rho) = \sum_k \operatorname{Tr}(\Pi_k \rho)\,\sigma_k$ for a projective measurement $\{\Pi_k\}$ and states $\sigma_k$. Find its Choi operator and show that it is a replacement channel exactly when all $\sigma_k$ are equal.

:::{dropdown} Solution
$\mathcal E(|i\rangle\langle j|) = \sum_k \langle j|\Pi_k|i\rangle\,\sigma_k$, so $J = \sum_k (\sum_{ij}\langle j|\Pi_k|i\rangle|i\rangle\langle j|) \otimes \sigma_k = \sum_k \Pi_k^{T} \otimes \sigma_k$. If all $\sigma_k = \sigma$, then $J = (\sum_k \Pi_k)^{T} \otimes \sigma = I \otimes \sigma$. Conversely, if $J = I \otimes \sigma$, apply the channel to the state $\Pi_k/\operatorname{Tr}\Pi_k$: the output is $\sigma_k$ by the definition and $\sigma$ by the theorem, so $\sigma_k = \sigma$ for every $k$.
:::

## Check your understanding

**1. In the process with $p = 1/2$, what is $p(b=0 \mid do(|0\rangle))$?**

- **A.** 1
- **B.** 1/2
- **C.** 1/4
- **D.** 3/4

:::{dropdown} Show the answer
**D. 3/4**

Replacement keeps only the direct-cause dependence: B gets $|0\rangle$ with probability $1 - p$ and $I/2$ with probability $p$, so $p(b=0) = 1 - p/2 = 3/4$. The value 1 is the conditional probability, which also counts the common cause.
:::

**2. Which statement characterizes a channel whose output does not depend on its input?**

- **A.** It is a unitary channel
- **B.** Its Choi operator is $I \otimes \sigma$
- **C.** It has a single Kraus operator
- **D.** Its Choi operator is a pure state

:::{dropdown} Show the answer
**B. Its Choi operator is $I \otimes \sigma$**

By the theorem, $\mathcal E(\rho) = \sigma$ for every state exactly when $J_{\mathcal E} = I \otimes \sigma$, and then $\mathcal E = R_\sigma$.
:::

**3. A measures $Z$ on its input and discards the outcome. What happens to downstream statistics compared with A doing nothing?**

- **A.** They never change, because no outcome was read
- **B.** They can change, because the measurement removes coherence
- **C.** They change only if the outcome is later read
- **D.** They change only for entangled inputs

:::{dropdown} Show the answer
**B. They can change, because the measurement removes coherence**

The non-selective measurement is the dephasing channel, which is not the identity. For the input $|+\rangle$ sent to an $X$ measurement, it lowers $p(+)$ from 1 to $1/2$ even though no one reads the outcome.
:::

**4. In a classical randomized experiment, why does $p(y \mid do(x))$ equal $p(y \mid x)$?** *(Review of [L4-06](#lesson-l4-06))*

- **A.** Randomization increases the sample size
- **B.** Randomization makes $Y$ independent of $X$
- **C.** Randomization makes $X$ a collider
- **D.** Randomizing $X$ removes every arrow into $X$, so no common cause links $X$ and $Y$

:::{dropdown} Show the answer
**D. Randomizing $X$ removes every arrow into $X$, so no common cause links $X$ and $Y$**

When the experimenter's coin sets $X$, nothing else causes $X$, so the association between $X$ and $Y$ is entirely causal. The quantum replacement $R_\sigma$ plays the same role: it removes the dependence of A's output on anything that came before.
:::

## Common misconceptions

- **"Measuring a qubit and ignoring the result is a passive observation."** The non-selective measurement is a channel, generally different from the identity. It removes coherence in the measured basis and can change every downstream statistic, as in the $|+\rangle$ example.
- **"Conditioning on A's outcome is the quantum version of do."** Conditioning selects runs and keeps the influence of any common cause; in the lesson's process it gives 1 where the interventional value is $1 - p/2$. The analogue of do is a replacement, which removes the dependence of A's output on its input.
- **"A local operation at A can change B's marginal statistics through an entangled state."** Without a direct-cause path, no channel at A changes B's reduced state (the no-signaling proposition). Only selecting A's outcomes changes B's conditional statistics.

:::{caution} Scope and limits
All results are for one specific process (a probabilistic mixture of a $|\Phi^+\rangle$ common cause and an identity direct cause, A's input maximally mixed), Lüders instruments, and exact probabilities with no finite-sample analysis. The Choi convention is input first with a transpose in the recovery rule. Replacement is presented as the analogue of Pearl's do, not as the unique quantum intervention; randomized repreparation and general instruments are also interventions, and which operations are available is part of the model. Coherent superpositions of causal structures are not treated.
:::

:::{seealso} Where this leads
The sign flip of the $Y$ correlation under a common cause is the key to [L6-05](#lesson-l6-05), where passive observation sometimes identifies the causal structure. Instruments and their Choi operators are packaged into process matrices in [L6-07](#lesson-l6-07), and counterfactual questions built on interventions return in [L6-09](#lesson-l6-09).
:::

:::{dropdown} For instructors

**Why this lesson.** Students who know the do-calculus tend to import "observation is free" into quantum settings. A single process in which three operations give three different numbers makes the distinction concrete and prepares the passive-observation result of the next lesson [@costa; @ried2015; @pearl].

**Teaching notes.**

- Ask students to predict $p(b=0)$ under replacement before computing; most predict 1 or $1/2$, and the answer $3/4$ forces them to separate the two branches.
- Stress that the Lüders instrument is a modeling choice: a different instrument with the same outcome probabilities (for example measure and reprepare a fixed state) gives different conditional states downstream.
- Use the Bloch animation to connect the $Y^{T} = -Y$ sign to the mirror image at $p = 1$; this is the geometric fact behind the sign criterion of Ried et al., the sign of the product of the three correlation coefficients.
- The stretch exercise on $p = 1/2$ is a quantum example of unfaithfulness: zero correlation with a nonzero causal effect.

**Assessment.** The worked example and the first challenge exercise test whether students can move between Bloch-vector reasoning and probabilities; the theorem's converse direction tests linear-algebra fluency.

**Going further.** Read the causal-tomography section of Ried et al. [@ried2015] and reconstruct the process from replacement data with six preparations and three measurements. Compare with the process-operator formalism of Costa and Shrapnel [@costa].

:::

## References

This lesson draws on [@costa; @ried2015; @watrous; @leifer; @pearl; @nielsenchuang].

**Further reading.**

- [@allen]: Quantum common causes and a quantum Markov condition built on unitary dilations.
- [@barrett]: Quantum causal models built on unitary circuits, with a quantum Markov condition and quantum conditional independence.

**Next:** [L6-05 · Discovery and informational access](#lesson-l6-05)
